# Significancia estadística entre modelos

Compara las 3 semillas de cada configuración sobre el mAP@50-95.

- **Entre arquitecturas distintas** (15 pares): test de Welch no pareado.
- **YOLO11m A vs B**: test t pareado por semilla (misma inicialización).
- Se reporta también el tamaño del efecto (d de Cohen).

In [ ]:
import json
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

RUNS = Path("../runs")
FIG_DIR = Path("figuras_significancia")
FIG_DIR.mkdir(exist_ok=True)
SEMILLAS = (0, 1, 2)

CONFIGS = [
    ("YOLO11m (A)",  "yolo_a_final"),
    ("YOLO11m (B)",  "yolo_b_final"),
    ("Faster R-CNN", "frcnn_final"),
    ("SSD300",       "ssd_final"),
    ("DETR",         "detr_final"),
    ("CNN baseline", "cnnb_final"),
]

MODEL_COLORS = {
    "YOLO11m (A)":  "#2a78d6",
    "YOLO11m (B)":  "#eb6834",
    "Faster R-CNN": "#1baf7a",
    "SSD300":       "#eda100",
    "DETR":         "#e87ba4",
    "CNN baseline": "#008300",
}
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID_COLOR = "#e1e0d9"
BASELINE_COLOR = "#c3c2b7"
SURFACE = "#fcfcfb"
STATUS_GOOD = "#0ca30c"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": BASELINE_COLOR, "axes.labelcolor": INK_SECONDARY,
    "axes.titlecolor": INK_PRIMARY, "text.color": INK_PRIMARY,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "grid.color": GRID_COLOR, "grid.linewidth": 0.8,
    "axes.grid": True, "axes.grid.axis": "x", "axes.axisbelow": True,
    "font.size": 10.5, "figure.dpi": 110,
})


def buscar_json(run_name: str):
    cands = [p / run_name / "eval_results.json" for p in RUNS.glob("*") if p.is_dir()]
    for c in cands:
        if c.exists():
            return c
    for pat in (f"**/{run_name}/eval_results.json", f"**/{run_name}/weights/eval_results.json"):
        hit = sorted(RUNS.glob(pat))
        if hit:
            return hit[0]
    return None


def cargar_crudos(prefijo: str, clave: str) -> list:
    # Valores de las 3 semillas para `clave` (p.ej. 'map50_95'), sin agregar.
    vals = []
    for s in SEMILLAS:
        p = buscar_json(f"{prefijo}_s{s}")
        if p is None:
            continue
        d = json.loads(p.read_text(encoding="utf-8"))
        v = d.get(clave)
        if v is not None:
            vals.append(v)
    return vals


def cohens_d_independiente(a, b):
    a, b = np.asarray(a), np.asarray(b)
    n1, n2 = len(a), len(b)
    s1, s2 = a.std(ddof=1), b.std(ddof=1)
    sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.inf


def cohens_d_pareado(diffs):
    diffs = np.asarray(diffs)
    return diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else np.inf


print("Utilidades cargadas.")

## Datos crudos (3 semillas por configuración, mAP@50-95)

In [ ]:
METRICA = "map50_95"   # metrica principal del protocolo (10 clases)
crudos = {etq: cargar_crudos(prefijo, METRICA) for etq, prefijo in CONFIGS}

filas = []
for etq, vals in crudos.items():
    if len(vals) < 2:
        continue
    arr = np.array(vals)
    media, s = arr.mean(), arr.std(ddof=1)
    t_crit = stats.t.ppf(0.975, df=len(arr) - 1)
    ic = t_crit * s / np.sqrt(len(arr))
    filas.append({"Modelo": etq, "n": len(arr), "valores": vals,
                  "media": media, "desv": s, "IC95_low": media - ic, "IC95_high": media + ic})

df_ic = pd.DataFrame(filas).set_index("Modelo")
df_ic[["n", "media", "desv", "IC95_low", "IC95_high"]].round(3)

## Figura 1 — Forest plot: mAP@50-95 con intervalo de confianza al 95%

Si los intervalos de dos modelos no se solapan, la diferencia es significativa
al 95% "a ojo" (equivalente aproximado a un test de dos muestras). Con n=3 el
IC es ancho — es la propia potencia baja haciéndose visible.

In [ ]:
orden = df_ic.sort_values("media", ascending=True).index.tolist()

fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(orden))
for i, m in enumerate(orden):
    row = df_ic.loc[m]
    ax.errorbar(row["media"], i, xerr=[[row["media"] - row["IC95_low"]],
                                        [row["IC95_high"] - row["media"]]],
                fmt="o", color=MODEL_COLORS[m], markersize=8, capsize=4,
                elinewidth=1.8, markeredgecolor="white", markeredgewidth=0.8)
    for v in row["valores"]:
        ax.scatter(v, i, marker="|", color=INK_MUTED, s=120, zorder=1)

ax.set_yticks(y)
ax.set_yticklabels(orden)
ax.set_xlabel("mAP@50-95 (%) — punto = media, barra = IC 95%, marcas = las 3 semillas")
ax.set_title("mAP@50-95 por modelo, con intervalo de confianza al 95%")
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_forest_plot_map.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "01_forest_plot_map.pdf", bbox_inches="tight")
plt.show()

## Comparaciones por pares — test de Welch (no pareado)

Los 15 pares posibles entre las 6 configuraciones. `p<0.05` marca diferencia
significativa; `d` es el tamaño del efecto de Cohen (>0.8 se considera grande,
independientemente de si el test alcanza significancia con esta n).

In [ ]:
filas = []
for (m1, p1), (m2, p2) in combinations(CONFIGS, 2):
    a, b = crudos.get(m1, []), crudos.get(m2, [])
    if len(a) < 2 or len(b) < 2:
        continue
    t, pval = stats.ttest_ind(a, b, equal_var=False)
    d = cohens_d_independiente(a, b)
    filas.append({
        "Modelo A": m1, "Modelo B": m2,
        "media A": np.mean(a), "media B": np.mean(b),
        "diferencia": np.mean(a) - np.mean(b),
        "t": t, "p": pval, "Cohen_d": d,
        "significativo (p<0.05)": pval < 0.05,
    })
df_pares = pd.DataFrame(filas)
df_pares.round(4)

## Figura 2 — Matriz de p-valores (Welch, mAP@50-95)

Verde = diferencia significativa (p<0.05), gris = no significativa con esta n.

In [ ]:
etqs = [e for e, _ in CONFIGS]
mat_p = pd.DataFrame(np.nan, index=etqs, columns=etqs)
for _, r in df_pares.iterrows():
    mat_p.loc[r["Modelo A"], r["Modelo B"]] = r["p"]
    mat_p.loc[r["Modelo B"], r["Modelo A"]] = r["p"]

fig, ax = plt.subplots(figsize=(7.5, 7))
mask_sig = mat_p < 0.05
colores = np.where(mat_p.isna(), np.nan, np.where(mask_sig, 1, 0))
cmap = plt.matplotlib.colors.ListedColormap([GRID_COLOR, STATUS_GOOD])
im = ax.imshow(colores.astype(float), cmap=cmap, vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(etqs)))
ax.set_xticklabels(etqs, rotation=35, ha="right")
ax.set_yticks(range(len(etqs)))
ax.set_yticklabels(etqs)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)

for i in range(len(etqs)):
    for j in range(len(etqs)):
        v = mat_p.iloc[i, j]
        if np.isnan(v):
            continue
        color_txt = "white" if v < 0.05 else INK_PRIMARY
        etiqueta = "<0.001" if v < 0.001 else f"{v:.3f}"
        ax.text(j, i, etiqueta, ha="center", va="center", fontsize=8, color=color_txt)

ax.set_title("p-valor (Welch) por par de modelos — mAP@50-95")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_matriz_pvalores.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "02_matriz_pvalores.pdf", bbox_inches="tight")
plt.show()

## Caso especial — YOLO11m variante A vs variante B (test pareado)

Único par del estudio donde el pareamiento por semilla es válido: misma
arquitectura, mismo seed (misma inicialización de pesos), solo cambia la
augmentación. Se usa un test t pareado, más potente que Welch con esta n.

In [ ]:
a = np.array(crudos["YOLO11m (A)"])
b = np.array(crudos["YOLO11m (B)"])
assert len(a) == len(b) == 3, "Hacen falta las 3 semillas de A y B para el test pareado"

diffs = a - b
t_pareado, p_pareado = stats.ttest_rel(a, b)
d_pareado = cohens_d_pareado(diffs)

print(f"mAP@50-95   A = {a.mean():.2f} ± {a.std(ddof=1):.2f}   "
      f"B = {b.mean():.2f} ± {b.std(ddof=1):.2f}")
print(f"Diferencia por semilla (A - B): {diffs.round(2).tolist()}")
print(f"Diferencia media: {diffs.mean():.2f} pp   (todas las semillas van en el mismo sentido: "
      f"{'sí' if (diffs > 0).all() or (diffs < 0).all() else 'no'})")
print(f"t pareado = {t_pareado:.3f}   p = {p_pareado:.4f}   Cohen's d pareado = {d_pareado:.2f}")
veredicto = "SIGNIFICATIVA" if p_pareado < 0.05 else "no alcanza significancia (n=3, df=2)"
print(f"\nVeredicto: la augmentación completa (A) produce una diferencia {veredicto} "
      f"frente a solo flip (B), en mAP@50-95.")

In [ ]:
for p in sorted(FIG_DIR.glob("*.pdf")):
    print(p.name)